# A Chatbot for a Company's Internal Documents

Employees ask the same questions again and again: who approves a business trip, how to reset a password, whether a taxi is reimbursed. The answers are in internal documents nobody remembers. A chatbot can answer from those documents, if it can find the right passage first.

That finding step is retrieval, the R in RAG (retrieval-augmented generation): the question is matched with passages of the documents, and only those passages go to a language model to write the answer. If retrieval picks the wrong passage, the answer will be wrong however good the model is. So this notebook builds and measures the retrieval:

1. split the documents into passages,
2. search them three ways: by meaning (embeddings), by words (BM25), and both combined,
3. check each against 32 test questions whose right document is known,
4. assemble the prompt a language model would get.

**Data:** [docs/](docs/), 12 made-up internal documents of a fictional company (IT, HR, finance, office), and [test_questions.json](test_questions.json), generated by [make_docs.py](make_docs.py).

In [1]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from fastembed import TextEmbedding

In [2]:
documents = {p.name: p.read_text(encoding="utf-8") for p in sorted(Path("docs").glob("*.md"))}
for name, text in documents.items():
    print(f"{name:<24} {text.splitlines()[0][2:]}")

fin_corporate_card.md    The corporate card
fin_expenses.md          Expense reimbursement
hr_business_trip.md      Business trips
hr_onboarding.md         The first week for new employees
hr_sick_leave.md         Sick leave
hr_vacation.md           Vacation
it_email_archive.md      Archiving old email
it_password.md           Resetting a forgotten password
it_service_desk.md       The IT service desk
it_vpn.md                Connecting to the office network from home (VPN)
office_badge.md          Access badges and visitors
office_parking.md        Parking


## 1. Passages

Each document is split at blank lines into paragraphs, and the document's title is kept with every passage, so a passage like "Submit a trip request..." still says it's about business trips.

In [3]:
passages = []
for name, text in documents.items():
    title, *paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    for paragraph in paragraphs:
        passages.append({"document": name, "text": f"{title.lstrip('# ')}. {paragraph}"})
print(f"{len(passages)} passages")
passages[0]

27 passages


{'document': 'fin_corporate_card.md',
 'text': 'The corporate card. Employees who travel often can get a corporate card. Ask your manager to request it in the finance portal.'}

## 2. Three ways to search

**By meaning.** A small embedding model (BAAI/bge-small-en-v1.5, which runs locally) turns each passage into 384 numbers that describe its meaning. A question is turned into numbers the same way, and the passages with the most similar numbers win, even if they use different words ("money back" and "reimbursed").

In [4]:
model = TextEmbedding("BAAI/bge-small-en-v1.5")
vectors = np.array(list(model.embed([p["text"] for p in passages])))
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

def search_meaning(question):
    q = next(iter(model.query_embed(question)))
    return vectors @ (q / np.linalg.norm(q))

**By words.** BM25 scores a passage by the question's words it contains, giving more weight to rare words and to passages where the word makes up more of the text.

In [5]:
def words(text):
    return re.findall(r"\w+", text.lower())

docs_words = [Counter(words(p["text"])) for p in passages]
lengths = np.array([sum(c.values()) for c in docs_words])
document_frequency = Counter(w for c in docs_words for w in c)
idf = {w: math.log(1 + (len(passages) - n + 0.5) / (n + 0.5)) for w, n in document_frequency.items()}

def search_words(question, k1=1.5, b=0.75):
    scores = np.zeros(len(passages))
    for w in set(words(question)):
        if w in idf:
            tf = np.array([c[w] for c in docs_words])
            scores += idf[w] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * lengths / lengths.mean()))
    return scores

**Both.** The two rankings are merged by reciprocal rank fusion: each passage gets 1 / (60 + its rank) from each list, so one near the top of either list rises, and one near the top of both rises most.

In [6]:
def search_both(question):
    fused = np.zeros(len(passages))
    for scores in (search_meaning(question), search_words(question)):
        ranks = np.empty(len(passages))
        ranks[np.argsort(-scores)] = np.arange(1, len(passages) + 1)
        fused += 1 / (60 + ranks)
    return fused

## 3. Measured on test questions

32 test questions: 20 that use the documents' own words, and 12 paraphrased ones that say the same thing differently ("Can I keep some days off for next year?" for carrying over vacation). For each, does the right document come first (hit@1), or among the top 3 passages (hit@3)?

In [7]:
tests = pd.DataFrame(json.loads(Path("test_questions.json").read_text(encoding="utf-8")))
methods = {"by meaning": search_meaning, "by words": search_words, "both": search_both}

def evaluate(methods, passages_used):
    rows, misses = [], []
    for name, search in methods.items():
        for t in tests.itertuples():
            top = [passages_used[i]["document"] for i in np.argsort(-search(t.question))[:3]]
            rows.append({"method": name, "questions": t.kind, "hit@1": top[0] == t.document, "hit@3": t.document in top})
            if top[0] != t.document:
                misses.append((name, t.question, t.document, top[0]))
    return pd.DataFrame(rows).groupby(["questions", "method"])[["hit@1", "hit@3"]].mean(), misses

scores, misses = evaluate(methods, passages)
scores

hit@1     hit@3
questions                 method                        
paraphrased               both        0.500000  0.833333
                          by meaning  0.833333  1.000000
                          by words    0.416667  0.500000
uses the documents' words both        1.000000  1.000000
                          by meaning  1.000000  1.000000
                          by words    1.000000  1.000000

In [8]:
pd.DataFrame(misses, columns=["method", "question", "right document", "found first"])

,method,question,right document,found first
0,by meaning,"I paid for a client dinner myself, how do I ge...",fin_expenses.md,fin_corporate_card.md
1,by meaning,A friend is coming to see me at work tomorrow,office_badge.md,hr_sick_leave.md
2,by words,Who signs off on flying to a conference in ano...,hr_business_trip.md,fin_corporate_card.md
3,by words,How much can I spend on lunch per day when I'm...,hr_business_trip.md,it_password.md
4,by words,I typed the wrong login too many times and now...,it_password.md,office_badge.md
5,by words,Outlook says I'm running out of space,it_email_archive.md,office_parking.md
6,by words,"I have the flu, do I need paperwork from a cli...",hr_sick_leave.md,hr_vacation.md
7,by words,"I paid for a client dinner myself, how do I ge...",fin_expenses.md,fin_corporate_card.md
8,by words,A friend is coming to see me at work tomorrow,office_badge.md,it_password.md
9,both,Who signs off on flying to a conference in ano...,hr_business_trip.md,fin_corporate_card.md


### Does keeping the title with each passage matter?

The same test with passages that don't start with their document's title:

In [9]:
bare = [{"document": p["document"], "text": p["text"].split(". ", 1)[1]} for p in passages]
vectors_titled, passages_titled, docs_words_titled = vectors, passages, docs_words
vectors = np.array(list(model.embed([p["text"] for p in bare])))
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
passages = bare
docs_words = [Counter(words(p["text"])) for p in passages]
lengths = np.array([sum(c.values()) for c in docs_words])
document_frequency = Counter(w for c in docs_words for w in c)
idf = {w: math.log(1 + (len(passages) - n + 0.5) / (n + 0.5)) for w, n in document_frequency.items()}
scores_bare, _ = evaluate(methods, passages)

# back to the titled passages for the rest of the notebook
vectors, passages, docs_words = vectors_titled, passages_titled, docs_words_titled
lengths = np.array([sum(c.values()) for c in docs_words])
document_frequency = Counter(w for c in docs_words for w in c)
idf = {w: math.log(1 + (len(passages) - n + 0.5) / (n + 0.5)) for w, n in document_frequency.items()}

pd.concat({"with title": scores["hit@1"], "without title": scores_bare["hit@1"]}, axis=1)

with title  without title
questions                 method                               
paraphrased               both          0.500000       0.500000
                          by meaning    0.833333       0.750000
                          by words      0.416667       0.333333
uses the documents' words both          1.000000       1.000000
                          by meaning    1.000000       0.950000
                          by words      1.000000       0.950000

## 4. The prompt for the language model

The top passages go to a language model together with the question and an instruction to answer only from them and say which document the answer comes from. Any chat model can take this prompt; the call itself needs an API key, so it isn't run here.

In [10]:
def build_prompt(question, k=3):
    top = np.argsort(-search_both(question))[:k]
    context = "\n\n".join(f"[{passages[i]['document']}]\n{passages[i]['text']}" for i in top)
    return (f"Answer the employee's question using only the passages below. Name the document you used. "
            f"If the passages don't contain the answer, say so.\n\n{context}\n\nQuestion: {question}")

print(build_prompt("Who approves a business trip abroad?"))

Answer the employee's question using only the passages below. Name the document you used. If the passages don't contain the answer, say so.

[hr_business_trip.md]
Business trips. Submit a trip request in the HR portal at least 10 working days before departure, with the purpose, dates, and estimated cost. Your direct manager approves trips within the country; trips abroad also need the department head's approval. Once approved, the travel team books flights and hotels.

[hr_business_trip.md]
Business trips. A business trip needs approval before any tickets are booked.

[hr_business_trip.md]
Business trips. The daily allowance for meals is 50 euros within the country and 70 euros abroad.

Question: Who approves a business trip abroad?


## Results

- When a question uses the documents' own words, every method finds the right document first, every time.
- Paraphrased questions are the real test. Search by meaning puts the right document first for 10 of 12 (83%) and in the top 3 for all of them; search by words manages 5 of 12 (42%), because the question and the document share no words ("Outlook says I'm running out of space" against "mailboxes are limited to 50 GB").
- Combining the two with equal weight gives 50% on paraphrased questions: when word search has nothing to go on, its ranking drags the meaning search down. Here meaning search alone, or a combination that leans on it, works best; word search earns its place on exact names, codes, and numbers, which these documents barely contain.
- Keeping the document title with each passage helps: without it, search by meaning drops from 83% to 75% on paraphrased questions.
- The misses are understandable: a client dinner paid personally matched the corporate card document instead of expenses, and a friend's visit matched sick leave instead of visitor registration. More test questions like these are the way to tune a chatbot before employees use it.

## Tools

Python, FastEmbed (BAAI/bge-small-en-v1.5), NumPy, pandas